In [41]:
import requests
import json
import pandas as pd
import googlemaps
from datetime import datetime
import numpy as np
from math import radians, cos, sin, asin, sqrt



In [46]:
house = pd.DataFrame({
    'id': [1],
    'latitude': [40.1930597],
    'longitude': [44.5022556]
})

df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'latitude': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'longitude': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})


## Calculate using OpenRouteService

In [47]:
for house_idx, house_row in house.iterrows():
    # Set the start coordinates to the current house coordinates
    start_lon = house_row['longitude']
    start_lat = house_row['latitude']
    
    # Initialize the closest distance and closest station ID to large values
    closest_distance = float('inf')
    closest_station_id = -1
    
    # Loop through each row in the df table
    for df_idx, df_row in df.iterrows():
        # Set the end coordinates to the current metro station coordinates
        end_lon = df_row['longitude']
        end_lat = df_row['latitude']
        
        # Define the API URL and parameters
        url = 'https://api.openrouteservice.org/v2/directions/foot-walking'
        params = {
            'api_key': '***REMOVED-ORS-API-KEY***',
            'start': f'{start_lon},{start_lat}',
            'end': f'{end_lon},{end_lat}'
        }

        # Make the API request
        response = requests.get(url, params=params)

        # Parse the response JSON
        data = json.loads(response.text)

        # Extract the distance from the response
        distance = data['features'][0]['properties']['segments'][0]['distance']
        print(distance)
        # Update the closest distance and closest station ID if the current distance is smaller
        if distance < closest_distance:
            closest_distance = distance
            closest_station_id = df_row['id']
    
    # Store the closest station ID in the new column for the current house row
    house.loc[house_idx, 'closest_station_id'] = closest_station_id

# Save the updated house table to a new CSV file
house.to_csv('house_with_closest_station.csv', index=False)

1049.4
555.3
1718.4
2284.2
3346.7
5202.6
6604.9
7063.8
9601.3
6934.2


## Calculate using Google Routing API

In [37]:
import googlemaps
from datetime import datetime
import pandas as pd

# Function to calculate walking distance between two points using the Google Maps API
def get_walking_distance(start_lon, start_lat, end_lon, end_lat):
    # Set up the Google Maps API client
    gmaps = googlemaps.Client(key='***REMOVED-GOOGLE-API-KEY***')

    # Define the start and end coordinates
    start = (start_lat, start_lon)
    end = (end_lat, end_lon)

    # Make the API request
    now = datetime.now()
    directions = gmaps.directions(start, end, mode='walking', departure_time=now)

    # Extract the distance from the response
    distance = directions[0]['legs'][0]['distance']['value']

    # Return the distance in meters
    return distance


# Sample house and metro station coordinates
houses = pd.DataFrame({
    'id': [1],
    'lat': [40.1930597],
    'lon': [44.5022556]
})

stations = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'lat': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'lon': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

In [40]:

# Calculate the walking distance from each house to each metro station
for i, house in houses.iterrows():
    distances = []
    for j, station in stations.iterrows():
        distance = get_walking_distance(house['lon'], house['lat'], station['lon'], station['lat'])
        distances.append(distance)
    stations['distance'] = distances
    print(stations['distance'])
    closest_station = stations.loc[stations['distance'].idxmin(), 'station_name']
    closest_dist = stations.loc[stations['distance'].idxmin(), 'distance']
    houses.at[i, 'closest_station'] = closest_station
    
    houses.at[i, 'closest_distance'] = closest_dist

# Print the houses table with the closest metro station
print(houses)


0    1056
1     329
2    1703
3    2287
4    3377
5    5039
6    6747
7    7184
8    9611
9    6960
Name: distance, dtype: int64
   id       lat        lon closest_station  closest_distance
0   1  40.19306  44.502256      Baghramyan             329.0


In [43]:
dist_matrix = np.zeros((len(houses), len(stations)))
for i in range(len(houses)):
    house = houses.iloc[i]
    distances = []
    for j in range(len(stations)):
        station = stations.iloc[j]
        distance = get_walking_distance(house['lon'], house['lat'], station['lon'], station['lat'])
        distances.append(distance)
    dist_matrix[i] = distances

# Find the index of the closest metro station for each house
closest_station_indices = np.argmin(dist_matrix, axis=1)

# Add the closest metro station to the houses table
houses['closest_station'] = stations.iloc[closest_station_indices]['id'].values

# Print the houses table with the closest metro station
print(houses)

   id       lat        lon  closest_station  closest_distance
0   1  40.19306  44.502256                2             329.0


## Calculating distance based on haversine formula

In [41]:
# Load the houses and stations tables into pandas dataframes
houses = pd.DataFrame({
    'house_id': [1, 2],
    'latitude': [40.1930597, 40.197446],
    'longitude': [44.5022556, 44.493467]
})

stations = pd.DataFrame({
    'station_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'latitude': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'longitude': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

# Define a function to calculate the Haversine distance between two points
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

# Loop through each house and station to find the shortest distance
distances = []
station_ids = []
for house_idx, house in houses.iterrows():
    shortest_distance = float('inf')
    nearest_station_name = ''
    for station_idx, station in stations.iterrows():
        distance = haversine(house['longitude'], house['latitude'], station['longitude'], station['latitude'])
        if distance < shortest_distance:
            shortest_distance = distance
            nearest_station_name = station['station_id']
    distances.append(shortest_distance)
    station_ids.append(nearest_station_name)

# Add the new columns to the houses table
houses['shortest_distance'] = distances
houses['nearest_station_id'] = station_ids

In [48]:
distances = []
for i, house in houses.iterrows():
    print(stations['station_name'][house['nearest_station_id']-1])
    distance = get_walking_distance(house['longitude'], house['latitude'], stations['longitude'][house['nearest_station_id']-1], stations['latitude'][house['nearest_station_id']-1])
    print(distance)
    distances.append(distance)
houses["walk_distance"] = distances
houses

Baghramyan
329
Barekamutyun
0


,house_id,latitude,longitude,shortest_distance,nearest_station_id,walk_distance
0,1,40.193060,44.502256,0.19109,2,329
1,2,40.197446,44.493467,0.00000,1,0


## Yerevan metro thingy

In [56]:
h1 = pd.read_excel('./data/Yerevan.xlsx')
houses = h1.head(100)

In [59]:
# Load the houses and stations tables into pandas dataframes

stations = pd.DataFrame({
    'station_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'x': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'y': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

# Define a function to calculate the Haversine distance between two points
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

# Loop through each house and station to find the shortest distance
#distances = []
station_ids = []
station_names = []
for house_idx, house in houses.iterrows():
    shortest_distance = float('inf')
    nearest_station_name = ''
    for station_idx, station in stations.iterrows():
        distance = haversine(house['y'], house['x'], station['y'], station['x'])
        if distance < shortest_distance:
            shortest_distance = distance
            nearest_station_id = station['station_id']
            nearest_station_name = station['station_name']
    #distances.append(shortest_distance)
    station_ids.append(nearest_station_id)
    station_names.append(nearest_station_name)

# Add the new columns to the houses table
#houses['shortest_distance'] = distances
houses['nearest_station_id'] = station_ids
houses['nearest_station_name'] = station_names

/var/folders/31/mgjvydd530n_q8xd9pnv0wrc0000gq/T/ipykernel_50804/3916827133.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  houses['nearest_station_id'] = station_ids
/var/folders/31/mgjvydd530n_q8xd9pnv0wrc0000gq/T/ipykernel_50804/3916827133.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  houses['nearest_station_name'] = station_names


In [60]:
distances = []
for i, house in houses.iterrows():
    distance = get_walking_distance(house['y'], house['x'], stations['y'][house['nearest_station_id']-1], stations['x'][house['nearest_station_id']-1])
    distances.append(distance)
houses["walk_distance"] = distances
houses

/var/folders/31/mgjvydd530n_q8xd9pnv0wrc0000gq/T/ipykernel_50804/3588086057.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  houses["walk_distance"] = distances


,index,id,price,rooms,square_meters,address,date,source,furniture,renovation,...,bathroom,rent_or_sale,links,y,x,district,shortest_distance,nearest_station_id,walk_distance,nearest_station_name
0,0,19136358,113000.0,2.0,55.0,"Աբելյան փողոց 22, Երևան",2023-03-13 18:58:27,list.am,0,No Renovation,...,1.0,sale appartments,http://list.am/en/item/19136358,44.472724,40.198827,Աջափնյակ,1.768436,1,2929,Barekamutyun
1,1,18934317,162000.0,3.0,90.0,"Lvovyan street 14, Yerevan",2023-03-13 18:58:28,list.am,0,Designer Renovation,...,1.0,sale appartments,http://list.am/en/item/18934317,44.562313,40.203270,Նոր Նորք,4.071285,3,6706,Yeritasardakan
2,2,19112565,35200.0,1.0,20.0,"Կոմիտասի պողոտա, Երևան",2023-03-13 18:58:28,list.am,0,Cosmetic Renovation,...,1.0,sale appartments,http://list.am/en/item/19112565,44.507210,40.206239,Արաբկիր,1.522578,1,1677,Barekamutyun
3,3,19061447,68999.0,1.0,42.0,"Khorenatsi street, Yerevan",2023-03-13 18:58:28,list.am,0,Cosmetic Renovation,...,1.0,sale appartments,http://list.am/en/item/19061447,44.515008,40.165975,Կենտրոն,0.542711,5,722,Zoravar Andranik
4,4,18995093,90000.0,2.0,50.0,"Komitas avenu 49, Երևան",2023-03-13 18:58:29,list.am,0,Euro Renovation,...,1.0,sale appartments,http://list.am/en/item/18995093,44.515673,40.206955,Արաբկիր,1.961833,2,2800,Baghramyan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,18945772,65000.0,4.0,124.0,"Պետրոս Դուրյան Թաղ. 3-րդ Փող., Երևան",2023-03-13 18:59:04,list.am,0,Partial Renovation,...,2.0,sale appartments,http://list.am/en/item/18945772,44.525213,40.157884,Էրեբունի,1.676013,6,2441,Sasuntsi Davit
96,96,17989774,215000.0,3.0,90.0,Yerevan › Kentron,2023-03-13 18:59:05,list.am,0,Major Renovation,...,1.0,sale appartments,http://list.am/en/item/17989774,44.514720,40.182989,Կենտրոն,0.507206,4,536,Republic Square
97,97,18546610,165000.0,3.0,83.0,"Աբելյան փողոց 777, Երևան",2023-03-13 18:59:05,list.am,0,Major Renovation,...,2.0,sale appartments,http://list.am/en/item/18546610,44.476775,40.199130,Աջափնյակ,1.430004,1,2684,Barekamutyun
98,98,19048460,295000.0,3.0,98.0,"Tsitsernakaberd Highway 1/2, Yerevan",2023-03-13 18:59:06,list.am,0,Designer Renovation,...,2.0,sale appartments,http://list.am/en/item/19048460,44.489343,40.178641,Կենտրոն,1.912216,2,3767,Baghramyan


# OpenSreetMap bus stop info

In [97]:
import requests

# Overpass API endpoint to get bus stop locations in Yerevan
url = "http://overpass-api.de/api/interpreter"

# Overpass QL query to retrieve all bus stops in Yerevan
query = """
[out:json];
area["name"="Yerevan"]->.a;
node(area.a)["highway"="bus_stop"];
out;
"""

# send HTTP request to Overpass API
response = requests.post(url, data=query)

# parse JSON response to get bus stop locations
json_response = response.json()
elements = json_response['elements']
print(elements)

bus_stops = []

for element in elements:
    # get bus stop name and coordinates
    name = element['tags']['name']
    lat = element['lat']
    lon = element['lon']

    # add bus stop to list
    bus_stop = {"name": name, "lat": lat, "lon": lon}
    bus_stops.append(bus_stop)

[]


In [94]:
bus_stops

[]